In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 26. LoRAtext text — text text text [CPU/GPU Benchmark ⑫]

> **Learning Goals**
> - LoRAtext text text text textdegreestext
> - text(Pruning)text text(sparsity)text text text
> - LoRA Trainingtext text text text text text text

## 26.1 PEFTtext text

text text: text text text → text GPU text, text.

**PEFT** (Parameter-Efficient Fine-Tuning): text text.
- LoRA, Adapter, Prefix Tuning, Prompt Tuning
- 1% text text text FTtext text Performance

## 26.2 LoRA — text text

text text $\Delta W$text text(low-rank)text text:
$$W' = W + \Delta W = W + B A$$

- $A \in \mathbb{R}^{r \times d}$, $B \in \mathbb{R}^{d \times r}$
- $r \ll d$ (text 4~64)
- text: $d^2 \to 2rd$ (text text)

text:
- $A \sim \mathcal{N}(0, \sigma^2)$ (text)
- $B = 0$ (text $\Delta W = 0$)

text: $\Delta W = \frac{\alpha}{r} B A$

Training: $W$text text, $A, B$text Training.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class LoRALinear(nn.Module):
    """LoRAtext Applicationtext LinearLayer."""
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # base Weight text
        self.base.weight.requires_grad = False
        if bias:
            self.base.bias.requires_grad = False
        # LoRA text
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        # base + LoRA delta
        base_out = self.base(x)
        lora_out = (x @ self.A.t()) @ self.B.t() * self.scaling
        return base_out + lora_out

# Parameter Count Comparison
d = 4096
full = nn.Linear(d, d)
lora = LoRALinear(d, d, r=8, alpha=16)
full_params = sum(p.numel() for p in full.parameters())
lora_params = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Full Linear: {full_params:,} params")
print(f"LoRA (r=8): {lora_params:,} trainable params")
print(f"Ratio: {lora_params/full_params*100:.3f}%")
print("\n=> LoRAtext 0.4% text Training!")


## 26.3 text $r$text text

$r$text text text ↑, text ↑. text textdegreestext text text $r$ text.
- text text: $r = 4$
- text text: $r = 16, 32, 64$


In [ ]:
# text text text Performance (text)
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
d = 4096
print(f"{'r':>5} | {'LoRA params':>12} | {'% of full':>10}")
print("-" * 35)
for r in ranks:
    lora_params = 2 * r * d  # A + B
    full_params = d * d
    print(f"{r:>5} | {lora_params:>12,} | {lora_params/full_params*100:>9.3f}%")

# Visualization
fig, ax = plt.subplots(figsize=(9, 5))
params_pct = [2 * r * d / (d * d) * 100 for r in ranks]
ax.plot(ranks, params_pct, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('Trainable params (%)')
ax.set_title(f'LoRA: rank vs trainable params (d={d})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch26_lora_ranks.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.4 text LoRA text

text LoRAtext text text:
- $W_q, W_k, W_v, W_o$ (Attention)
- FFNtext $W_1, W_2$

text Attentiontext Q, Vtext text (QLoRA text text). text text Performancetext text.


In [ ]:
# Attentiontext LoRA text
class LoRAAttention(nn.Module):
    def __init__(self, d_model, n_heads, r=8):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # base text (text)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_qkv.weight.requires_grad = False
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.W_o.weight.requires_grad = False
        # LoRA text (QKVtext)
        self.lora_qkv_A = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.lora_qkv_B = nn.Parameter(torch.zeros(3 * d_model, r))
        self.scaling = 16 / r

    def forward(self, x, mask=None):
        B, T, D = x.shape
        base = self.W_qkv(x)
        # LoRA delta
        delta = (x @ self.lora_qkv_A.t()) @ self.lora_qkv_B.t() * self.scaling
        qkv = base + delta
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# text
d, h = 64, 4
attn = LoRAAttention(d, h, r=4)
x = torch.randn(2, 10, d)
out, w = attn(x)
print(f"Output: {out.shape}")
n_trainable = sum(p.numel() for p in attn.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in attn.parameters())
print(f"Trainable: {n_trainable:,} / Total: {n_total:,} ({n_trainable/n_total*100:.2f}%)")


## 26.5 text (Pruning)

### text text
text text 0text:
$$\tilde{W}_{ij} = W_{ij} \cdot \mathbb{1}[|W_{ij}| > \tau]$$

Problem: text text text text.

### text text
text/text text text:
- text textdegreestext text text (attention head pruning)
- text/text text (CNN)
- text text

**Lottery Ticket Hypothesis**: text text text Performance text text text text.


In [ ]:
# text text
def magnitude_prune(W, sparsity=0.5):
    """Magnitude text textStructuretext Pruning."""
    threshold = torch.quantile(W.abs().flatten(), sparsity)
    mask = (W.abs() > threshold).float()
    return W * mask, mask

# Test
torch.manual_seed(0)
W = torch.randn(100, 100) * 0.1
for sparsity in [0.5, 0.7, 0.9, 0.95]:
    W_pruned, mask = magnitude_prune(W, sparsity)
    actual_sparsity = 1 - mask.mean()
    print(f"text sparsity {sparsity}: text {actual_sparsity:.4f}, "
          f"text Circletext {int(mask.sum())}text")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
W_orig = torch.randn(50, 50) * 0.1
for ax, sp in zip(axes, [0.0, 0.7, 0.9]):
    W_p, _ = magnitude_prune(W_orig, sp)
    ax.imshow(W_p.numpy(), cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.set_title(f'Sparsity = {sp}')
plt.tight_layout()
plt.savefig('../figures/ch26_pruning.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.6 [CPU/GPU Benchmark ⑫] Full FT vs LoRA vs QLoRA


In [ ]:
# Full FT vs LoRA Comparison
from llm_math.bench import time_fn

# text Model (text text)
class TinyModel(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

# Full FT
def make_full_ft():
    model = TinyModel()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

# LoRA
class TinyModelLoRA(nn.Module):
    def __init__(self, d=512, r=8):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
        # base text
        for p in self.parameters():
            p.requires_grad = False
        # LoRA text (base fc1/fc2/fc3text LoRA text text text
        # base text deltatext text text)
        self.lora_A1 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B1 = nn.Parameter(torch.zeros(d * 4, r))
        self.lora_A2 = nn.Parameter(torch.randn(r, d * 4) * 0.01)
        self.lora_B2 = nn.Parameter(torch.zeros(d, r))
        self.lora_A3 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B3 = nn.Parameter(torch.zeros(d, r))
        self.scaling = 16 / r
    def forward(self, x):
        # fc1 + LoRA delta
        h1 = F.relu(self.fc1(x) + (x @ self.lora_A1.t()) @ self.lora_B1.t() * self.scaling)
        # fc2 + LoRA delta
        h2 = F.relu(self.fc2(h1) + (h1 @ self.lora_A2.t()) @ self.lora_B2.t() * self.scaling)
        # fc3 + LoRA delta
        out = self.fc3(h2) + (h2 @ self.lora_A3.t()) @ self.lora_B3.t() * self.scaling
        return out

def make_lora():
    model = TinyModelLoRA(r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

# Training text text Time Comparison
def bench_step(model, opt, x, y, loss_fn, device='cpu'):
    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

d = 256
loss_fn = nn.MSELoss()
x = torch.randn(8, d)
y = torch.randn(8, d)

# Modeltext d=256text Generation
def make_full_ft(d=256):
    model = TinyModel(d=d)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

def make_lora(d=256):
    model = TinyModelLoRA(d=d, r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

model_full, opt_full = make_full_ft(d=d)
model_lora, opt_lora = make_lora(d=d)

n_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Full FT: {n_full:,} trainable params")
print(f"LoRA:    {n_lora:,} trainable params ({n_lora/n_full*100:.2f}%)")

res_full = time_fn(bench_step(model_full, opt_full, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
res_lora = time_fn(bench_step(model_lora, opt_lora, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
print(f"\nFull FT 1 step: {res_full['mean_ms']:.3f} ms")
print(f"LoRA 1 step:    {res_lora['mean_ms']:.3f} ms")
print(f"Speed Improvement: {res_full['mean_ms'] / res_lora['mean_ms']:.2f}x")
print("\n=> LoRAtext Training text text Memorytext Speed text text.")


## 26.7 Key Takeaways

| text | Training text | text | Performance |
|---|---|---|---|
| Full FT | 100% | text | text |
| LoRA | ~1% | text | text |
| QLoRA | ~1% + 4-bit base | text text | text text |
| Adapter | ~5% | text | text |
| Prefix Tuning | ~0.1% | text text | text text |

## Exercises

1. LoRA rank $r = 1, 4, 16, 64$text Trainingtext Performancetext Comparisontext.
2. LoRAtext QKV text text text Qtext text text Comparisontext.
3. 50%, 70%, 90% text Modeltext textdegreestext Comparisontext.
4. Full FT vs LoRAtext Training Timetext text text.
5. LoRAtext text text text text text text.

> Solutions: `solutions/ch26_solutions.ipynb`
